# AegisForecast â€” GPU Full Train + CIC-IDS2017 Transfer Validation

1. Full-scale training (60 scenarios, 25 epochs, seq_len 48, slope features)
2. Eval battery: FP rate + lead-time distribution
3. **CIC-IDS2017 zero-shot transfer validation** on real captured attacks
4. Export v2 checkpoint (state_dict + config metadata) for download

**Runtime:** Runtime â†’ Change runtime type â†’ **GPU (T4)**

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# Upload the repo (zip) through the Files panel, or clone if hosted:
#   !git clone https://github.com/TEAM-ORG/aegisforecast.git
import os, sys, zipfile, glob
if not os.path.isdir('aegisforecast'):
    zips = glob.glob('/content/*.zip')
    assert zips, 'upload aegisforecast.zip to /content first'
    with zipfile.ZipFile(zips[0]) as z: z.extractall('/content')
os.chdir('/content/aegisforecast')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
# Full-scale dataset: 60 scenarios + 14 benign sessions, seq 48, slope features
# (includes multi-host perimeter benign views by default — see build_dataset)
# Set MODEL = 'transformer' for the challenger variant.
MODEL = 'bilstm'  # or 'transformer'
import numpy as np
from aegis.features import FEATURE_DIM, SEQ_LEN
from aegis.ml.dataset import build_dataset

Xtr, Xva, Atr, Ava, Str, Sva, Htr, Hva = build_dataset(
    n_scenarios=60, n_benign=14, seed=7, seq_len=SEQ_LEN, use_slope=True)
print(f'train={len(Xtr)}  val={len(Xva)}  tensors {Xtr.shape[1]}x{Xtr.shape[2]}  attack-rate={Atr.mean():.2%}')


In [ ]:
# GPU training with the same loss/class-weighting as aegis.ml.train
import time, torch, torch.nn as nn, torch.nn.functional as F
from aegis.ml.model import BiLSTMAttention, TransformerForecaster, HORIZON_MAX_MIN
MODELS = {'bilstm': BiLSTMAttention, 'transformer': TransformerForecaster}
from aegis.ml.train import auc_score, evaluate

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS, LR, SEED = 25, 2e-3, 7
torch.manual_seed(SEED); np.random.seed(SEED)

model = MODELS[MODEL](input_dim=Xtr.shape[2], seq_len=Xtr.shape[1]).to(dev)
print(f'model={MODEL} params=' + f"{sum(p.numel() for p in model.parameters()):,}")
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

Xtr_t = torch.tensor(Xtr).to(dev); Atr_t = torch.tensor(Atr).to(dev)
Str_t = torch.tensor(Str).to(dev);  Htr_t = torch.tensor(Htr).to(dev)
n_pos = float((Atr > .5).sum()); n_neg = float((Atr <= .5).sum())
pos_w = float(np.clip(n_neg / max(n_pos, 1), 1.0, 5.0))
Wtr_t = Atr_t * pos_w + (1 - Atr_t)
print(f'pos_weight {pos_w:.2f} | params {sum(p.numel() for p in model.parameters()):,}')

t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xtr_t), device=dev)
    tot = 0.0
    for i in range(0, len(perm), 256):
        b = perm[i:i+256]
        fwd = model(Xtr_t[b])
        la = F.binary_cross_entropy(fwd['attack'], Atr_t[b], weight=Wtr_t[b])
        ls = ce(fwd['stage'], Str_t[b])
        m = Atr_t[b] > 0.5
        lh = mse(fwd['horizon'][m], Htr_t[b][m]) if m.any() else torch.tensor(0., device=dev)
        loss = la + 0.6 * ls + 0.4 * lh
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); tot += loss.item() * len(b)
    sched.step()
    pa, ps, ph = evaluate(model.to('cpu'), Xva, Ava, Sva, Hva); model = model.to(dev)
    m = Ava > 0.5
    hmae = np.abs(ph - Hva * HORIZON_MAX_MIN)[m].mean() if m.any() else float('nan')
    print(f'ep {ep:02d}  loss={tot/len(Xtr):.4f}  auc={auc_score(Ava, pa):.3f}  '
          f'stage_acc={(ps==Sva).mean():.3f}  horizon_mae={hmae:.1f}m  ({time.time()-t0:.0f}s)')

In [ ]:
# Eval battery on the GPU-trained model (FP + lead time)
model_cpu = model.to('cpu')
import torch
torch.save({'state_dict': model_cpu.state_dict(), 'seq_len': int(Xtr.shape[1]),
            'feature_dim': int(Xtr.shape[2]), 'use_slope': True,
            'model_name': MODEL},
           'aegis/ml/artifacts/forecast_model.pt')

from aegis.ml.evaluate import false_positive_rate, lead_time_distribution
mp = 'aegis/ml/artifacts/forecast_model.pt'
fp = false_positive_rate(mp, n_sessions=6, seed=4242)
lt = lead_time_distribution(mp, n_scenarios=12, seed=4243)
print(f"FP: {fp['alerts']} alerts / {fp['monitored_hours']}h -> {fp['alerts_per_hour']}/hr")
print(f"Lead: coverage {lt['coverage_pct']}%  median {lt['lead_median_min']}m  "
      f"p10 {lt['lead_p10_min']}m  p90 {lt['lead_p90_min']}m")

## CIC-IDS2017 transfer validation (zero-shot on real captured attacks)

In [ ]:
# Friday-PortScan subset via kagglehub (public mirror of CIC-IDS2017 flows)
import os
os.environ['KAGGLEHUB_CACHE'] = '/content/datasets'
try:
    import kagglehub
except ImportError:
    !pip -q install kagglehub
    import kagglehub

path = kagglehub.dataset_download('cicdataset/cic-ids2017')
print('dataset root:', path)
import glob
csvs = sorted(glob.glob(path + '/**/*.csv', recursive=True))
for c in csvs: print(' ', c)

In [ ]:
# Load the PortScan CSV (MachineLearningCVE Friday file) and transfer-evaluate
from pathlib import Path
from aegis.ml.cic_loader import (load_cic_csv, flows_to_windows,
                                 windows_to_sequences, transfer_evaluate)

portscan_csv = next((c for c in csvs if 'ortScan' in Path(c).stem or 'riday' in Path(c).stem), csvs[0])
print('using:', portscan_csv)
flows = load_cic_csv(Path(portscan_csv), max_rows=150_000)
print('flows:', len(flows), '| attack flows:', sum(f['_attack'] for f in flows))
windows, labels = flows_to_windows(flows)
print('windows:', len(windows), '| attack windows:', sum(labels))
X, y = windows_to_sequences(windows, labels)
res = transfer_evaluate(model_cpu, X, y)
print('TRANSFER RESULT (real CIC-IDS2017 PortScan):')
for k, v in res.items(): print(f'  {k}: {v}')

In [ ]:
# Persist metrics + download artifacts
import json
from google.colab import files
art = Path('aegis/ml/artifacts')
(art / 'metrics.json').write_text(json.dumps({
    'val_auc': round(auc_score(Ava, pa), 4),
    'val_stage_acc': round(float((ps == Sva).mean()), 4),
    'val_horizon_mae_min': round(float(np.abs(ph - Hva * HORIZON_MAX_MIN)[Ava > .5].mean()), 2),
    'n_train': len(Xtr), 'n_val': len(Xva),
    'seq_len': int(Xtr.shape[1]), 'feature_dim': int(Xtr.shape[2]),
    'use_slope': True, 'pos_weight': round(pos_w, 2), 'epochs': EPOCHS,
    'fp_alerts_per_hour': fp['alerts_per_hour'],
    'lead_coverage_pct': lt['coverage_pct'],
    'lead_median_min': lt['lead_median_min'],
    'cic_transfer': res,
}, indent=2))
files.download(str(art / 'forecast_model.pt'))
files.download(str(art / 'metrics.json'))
print('download both files -> place in aegis/ml/artifacts/ locally')